# Week 6: Diagnosis Drift Analysis

Loads `data/processed/drift_results_claude.jsonl` (per-trial results) and `drift_rate_table.json` (aggregated), produced by `src/eval_drift.py`.


In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt

with open('../data/processed/drift_results_claude.jsonl', encoding='utf-8') as f:
    results = [json.loads(line) for line in f]

with open('../data/processed/drift_rate_table.json', encoding='utf-8') as f:
    table = json.load(f)

df = pd.DataFrame(results)
print(f"{len(df)} drift trials loaded")
df.head()

## Overall Diagnosis Drift Rate

In [ ]:
print(f"Overall DDR: {table['overall_ddr']:.1%} (n={table['overall_n']})")

## DDR by adversarial note category

Each category tests a different hypothesized mechanism - see `docs/week3-taxonomy-and-vignettes.md` for what each one is designed to probe (tone bias, authority bias, anchoring, inference-from-action, informal-narrative bias).

In [ ]:
cat_df = pd.DataFrame(table['by_category']).T
cat_df = cat_df.sort_values('ddr', ascending=False)
cat_df

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(cat_df.index, cat_df['ddr'])
ax.set_ylabel('Diagnosis Drift Rate')
ax.set_title('DDR by adversarial note category')
ax.set_ylim(0, 1)
for i, v in enumerate(cat_df['ddr']):
    ax.text(i, v + 0.02, f'{v:.1%}', ha='center')
plt.tight_layout()
plt.savefig('../docs/week6-ddr-by-category.png', dpi=150)
plt.show()

## DDR by data source (MedQA vs. real MIMIC notes)

In [ ]:
source_df = pd.DataFrame(table['by_source']).T
source_df

## Spot-check a few individual trials

Worth checking a handful of the actual generated notes and before/after diagnoses since the grading is LLM-as-judge, and it can be pretty ambiguous when the model *adds* detail rather than cleanly changing its answer (see the known caveat about this in `docs/week6-drift-results.md`).

In [ ]:
for _, r in df.sample(5, random_state=0).iterrows():
    print('---')
    print(f"case: {r['case_id']} | category: {r['category']} | drifted: {r['drifted']}")
    print(f"before: {r['diagnosis_before']}")
    print(f"note: {r['adversarial_note']}")
    print(f"after: {r['diagnosis_after']}")